### Q29. MCP handshake 实现状态？

**回答框架：**
三种传输协议全部实现（P0-3 + df48161 提交补齐）：
- **stdio 传输**：JSON-RPC 握手已实现，通过子进程 stdin/stdout 通信
- **SSE 传输**：✅ 已实现 — 用 reqwest HTTP client + eventsource-stream 解析 SSE event stream。连接流程：POST initialize → parse SSE response → POST initialized → POST tools/list → parse SSE response
- **StreamableHttp 传输**：✅ 已实现 — 类似 SSE 但支持 session ID 管理，服务器在初始响应中返回 session ID，后续请求携带该 ID

**代码证据：** `mcp_real.rs:125` 的 McpTransport enum 包含 Stdio/Sse/StreamableHttp 三种变体。`mcp_real.rs:340-436` 的 SSE 连接逻辑和 `mcp_real.rs:438-524` 的 StreamableHttp 连接逻辑。McpConnection struct 有 `http_client`/`sse_url`/`post_url`/`session_id` 字段用于 SSE/StreamableHttp。

生产对接：现在可以直接连接远程 MCP server（如 Claude Desktop 的 filesystem server、社区发布的 MCP server），不需要中间代理。

---

### Q30. WASM 沙箱和 ShellTool regex 黑名单是替代还是并行？SandboxBackend 三种后端怎么选？

**回答框架：**
三层安全体系，**并行关系**：
- ShellTool 的 regex 黑名单是**快速拒绝层**——匹配 `rm -rf`、`mkfs` 等危险命令直接拒绝
- SandboxBackend 是**进程级隔离层**——wrap_command() 将 shell 命令包装在真实隔离环境中执行
- WASM 沙箱是**计算型隔离层**——对于需要执行但不属于系统命令的工具，在 WASM 中运行

SandboxBackend 三种后端（`sandbox.rs`）：
- **Seatbelt（macOS）**：用 sandbox-exec 实现文件系统+网络限制——和 Claude Code 使用同一机制
- **Docker（Linux）**：用容器实现完整进程隔离——和 Codex/OpenHands 使用同一机制
- **Regex（默认 fallback）**：增强版 regex + 工作目录限制，Windows 等不支持 Seatbelt/Docker 的平台使用

DomainPack 可以指定 per-tool sandbox policy（如 Coding 用 Seatbelt/Docker，Research 用 Regex 允许网络访问）。

选择逻辑：
- 系统级命令（ls、git、cargo）→ ShellTool + SandboxBackend 隔离
- 计算型任务（数据转换、格式化、代码分析）→ WasmTool（WASM 沙箱中执行，零 I/O 访问）
- 同一个功能如果有 Shell 版本和 WASM 版本 → DomainPack 可以选择用哪个（通过 tool_decorators 覆写描述）

**代码证据：** `sandbox.rs:67` 的 SandboxBackend trait，`sandbox.rs:97` SeatbeltBackend，`sandbox.rs:227` DockerBackend，`sandbox.rs:336` RegexBackend。`sandbox.rs:399` 的 `default_sandbox_backend()` 按平台自动选择。

---

### Q31. WasmTool 输入输出怎么传递？需要读文件时怎么办？

**回答框架：**
输入输出通过 WASM **线性内存**传递：
1. 序列化 args 为 JSON 字符串
2. 将 JSON bytes 写入 WASM 内存（固定偏移量 offset=8）
3. 调用 WASM 的 `execute(input_offset, input_len)` 函数
4. WASM 模块返回 `(output_ptr, output_len)` 指向输出在内存中的位置
5. 从内存中读取 output bytes，反序列化为 ToolOutput

需要读文件时：**当前不支持**——WASM 沙箱是零 I/O 访问（Wasmtime 的 linker 不注册任何 host function）。

未来扩展方向：注册 host function（如 `host_read_file(path) → content`），让 WASM 模块通过宿主函数间接访问文件。但这会削弱沙箱隔离性——需要权限控制。

**代码证据：** `wasm/src/tool.rs:109-207` 的 execute() 方法：写入 memory → call execute → 读 memory → parse_tool_output。没有 host function 注册。

---

### Q32. Wasmtime 编译延迟和内存开销？

**回答框架：**
Wasmtime 的编译延迟：首次加载 WASM 模块时需要编译（cranelift JIT），后续使用 module_cache 缓存。首次编译约 10-50ms（取决于模块大小），缓存后约 <1ms（只创建 Store 和 Instance）。

内存开销：WasmRuntime 有 `max_memory_pages` 配置（默认值在 config.rs），限制 WASM 模块的内存使用。fuel_limit 也限制了执行指令数。

257 个测试中 WASM 相关性能测试：**目前没有专门的 WASM 性能 benchmark 测试**——只有功能测试（WasmTool 的创建和 execute mock）。

---

### Q33. PermissionLevel 映射到具体工具？risk_level 和 permission_level 冲突时？

**回答框架：**
映射关系：
- **Read 级**：FileReadTool、GrepTool、GlobTool、FileListTool、EnvironmentTool
- **Standard 级**：FileEditTool、FileWriteTool、NotebookEditTool、MCP 交互
- **Full 级**：ShellTool、FileDeleteTool、WebFetchTool

冲突处理：**DomainPack 的 PermissionProfile 优先**。
- 解析链：deny_by_default → auto_approve → require_confirmation → permission_overrides → fallback to tool's risk_level()
- 如果 DomainPack 通过 permission_overrides 把 ShellTool 覆写为 Standard（比 tool 自身的 Full 更宽松）→ 这个覆写生效，但仅在该 domain 上下文中有效
- risk_level() 是 tool 级别默认值，permission_level() 是 domain 级别覆写——domain 优先

---

### Q34. ChannelApprovalGate 和 PlatformApprovalGate 的区别？

**回答框架：**
- **ChannelApprovalGate**：异步的——发送审批请求到 channel，等待外部回复。适用于 CLI 场景（用户在终端输入 y/n）
- **PlatformApprovalGate**：同步弹对话框——直接调用平台原生 UI（NSAlert/AlertDialog/UIAlertController）。适用于桌面/移动端应用场景

Android 上用 PlatformApprovalGate——因为 Android 的 AlertDialog 是 UI 层组件，不能从 channel 等待。

本质区别：Channel 是**异步消息模型**（发→等回），Platform 是**同步阻塞模型**（弹框→用户操作→返回）。Channel 更通用但需要 UI 层配合消费消息；Platform 更直接但绑定平台 UI 框架。

---

### Q35. deny_by_default 为什么优先级最高而不是最低？

**回答框架：**
安全哲学：**fail-safe default（安全兜底原则）**。

如果 deny 优先级低（最低）：
1. coding domain auto_approve shell → Shell 被批准
2. 然后 deny_by_default 检查 → 发现 shell 的 `rm -rf` args 应该 deny
3. 但 shell 已经被批准执行了！deny 来不及拦截

所以 deny 必须是最高优先级——**在任何批准逻辑之前先检查拒绝**，确保危险操作永远不会被执行。这和防火墙的"默认拒绝"策略同理：先排除已知危险，再考虑是否批准。

**代码证据：** `permission_profile.rs:181-206` 的 `resolve()` 方法，Step 1 先检查 deny patterns。

---

## 第五层：记忆、RAG 与评测

### Q36. STM 滑动窗口驱逐触发条件？

**回答框架：**
触发条件：**窗口满时驱逐最老的**。

具体实现（`short_term.rs:38-46`）：
```rust
pub fn push(&mut self, entry: MemoryEntry) -> Option<MemoryEntry> {
    let evicted = if self.entries.len() >= self.window_size {
        self.entries.pop_front()  // 驱逐最老的
    } else {
        None
    };
    self.entries.push_back(entry);
    evicted
}
```

驱逐策略是 FIFO（最老的先出），不是按重要性排序。因为 STM 的角色是"最近的对话上下文"，最近的自然最重要。

被驱逐的 entry 如果 `evict_to_ltm` 配置为 true（MemoryManagerConfig），会被存入 LTM 的 vector store + content store。

---

### Q37. HNSW 向量存储是自己实现的？和 hnswlib 比过吗？

**回答框架：**
当前实现是**简单的暴力 cosine similarity 搜索**，不是真正的 HNSW。

代码注释明确说（`long_term.rs:25-29`）：
"A simple in-memory vector store using brute-force cosine similarity search. Suitable for mobile deployment with no external dependencies. Can be swapped for a proper HNSW implementation."

和 hnswlib 没做过对比——因为当前实现就是 brute-force（O(N) 搜索），不适用于大规模数据。但 OneAI 的设计是 Agent 内部记忆，通常 N < 10000，brute-force 在这个规模下性能足够。

未来改进：EmbeddedVectorStore 有 `ThreadSafe` 包装，可以替换为真正的 HNSW 实现而不影响上层 LongTermMemory 的接口。

---

### Q38. 长期记忆"混合评分"是什么？

**回答框架：**
混合评分 = **语义相似度 × α + 时间近度 × β**

具体实现（`hybrid_scorer.rs`）：
```rust
pub struct HybridScorer {
    alpha: f32,  // 语义相似度权重（默认 0.7）
    beta: f32,   // 时间近度权重（默认 0.3）
}

score = alpha * semantic_score + beta * temporal_score
```

比例选择：默认 α=0.7、β=0.3，因为语义相关性比时间近度更重要（一条语义相关但较早的记忆，比一条语义不相关但刚写入的记忆更有价值）。

可通过 `HybridScorer::with_weights(alpha, beta)` 自定义比例。

---

### Q39. EmbeddingService 三种实现的适用场景？

**回答框架：**
- **FastEmbed（本地 ONNX）**：适用于桌面端/离线场景，零网络依赖，延迟低（~50ms），但模型较小（all-MiniLM-L6-v2），召回率中等
- **Ollama**：适用于本地 GPU 场景，可选择更大的 embedding 模型（nomic-embed-text），质量更高，但需要 Ollama 服务运行
- **OpenAI**：适用于服务端场景，text-embedding-3-small 质量最好，但需要网络和 API key，有成本

生产推荐：**服务端用 OpenAI，本地/移动端用 FastEmbed**。

FastEmbed 在移动端没有实际测试过——这是已知差距。ONNX Runtime 在 Android/iOS 上可以运行，但 OneAI 的 UniFFI binding 没有暴露 embedding service 到移动端。

---

### Q40. 分块策略 SentenceBoundary 和 FixedSize 实际差异？

**回答框架：**
- **FixedSize**：按固定 token 数分块，简单但可能切断句子
- **SentenceBoundary**：按句子边界分块，语义完整性好但块大小不均匀

实际差异：没有做过系统性对比测试——这是已知差距。理论上 SentenceBoundary 在检索召回率上应该更好（因为分块更符合语义边界），但块大小不均匀可能导致某些块信息密度低。

---

### Q41. Agent 的"成功"怎么定义？

**回答框架：**
当前定义比较简单：
- **DirectAnswer** 产生时 = 循环结束 = "任务完成"（不管回答质量）
- `AgentLoopResult.completed = true` 表示循环自然结束（不是被 hard_max_iterations 截断）

更精确的"成功"定义需要人工评估——OpenInference 轨迹日志记录的是工具调用成功率（工具执行是否报错），不是任务是否正确完成。

这是一个已知局限：**工具调用成功 ≠ 任务完成**。例如 Agent 可能成功调用了所有工具但给出了错误答案。

未来改进方向：引入 structured output validation（JSON Schema 验证最终答案格式）或人工标注的评测数据集。

---

### Q42. 轨迹日志和 OpenTelemetry trace/span 模型怎么对应？

**回答框架：**
映射关系：
- **一个 AgentLoop 执行 = 一个 AGENT span**（顶层 span）
- **每轮迭代 = 一个 WorkflowStep event**（在 AGENT span 内）
- **每轮 LLM 推理 = 一个 LLM span**（嵌套在迭代内）
- **每个工具调用 = 一个 TOOL span**（嵌套在迭代内）
- **子代理执行 = 一个 AGENT span**（嵌套在主 Agent span 内——形成父子关系）

子代理的 span 嵌套：通过 `parent_span_id` 关联——子代理的 AGENT span 的 parent 是主 Agent 的 LLM 或 TOOL span。

**代码证据：** `agent_loop.rs:720-727` 创建 `loop_span_id`（AGENT span），858-867 创建 `infer_span_id`（LLM span）。

---

### Q43. 渐进式 Checkpoint 的"渐进"是什么意思？

**回答框架：**
"渐进"指的是**保存策略**，不是存储后端升级：
- **EveryStep**：每轮迭代保存（最安全但开销大）
- **EveryNSteps**：每 N 轮保存（平衡安全和开销）
- **CriticalNodes**：只在关键节点保存（如范式切换、子代理委派后）

不是"只存增量"——当前实现是每轮保存完整的 LoopState（conversation + global_state + iterations + active_paradigm）。增量 checkpoint 是未来优化方向。

也不是"逐步升级后端"——Memory/SQLite/Postgres 是用户选择的存储后端，不是自动升级。

---

### Q44. SQLite 和 Postgres Checkpoint 后端实际实现了吗？

**回答框架：**
当前状态（竞品分析标注的差距）：
- **MemoryCheckpointBackend**：已实现（开发/测试用）
- **SQLiteCheckpointBackend**：**未完整实现**——竞品分析标注 "SqliteCheckpoint 未实现"，目前只有 FilePersistence
- **PostgresCheckpointBackend**：**未实现**

ProgressiveCheckpointManager 的 `list()` 方法也是 todo!()（竞品分析标注 "ProgressiveCheckpoint list todo!()"）。

实际可用的只有 Memory 后端。SQLite/Postgres 是规划中的生产级后端。

---

### Q45. 从任意检查点 fork 是什么意思？

**回答框架：**
fork 的含义：从一个历史检查点创建一个新的 AgentSession 分支，这个分支和原来的会话**共享 LTM 但 STM 独立**。

具体机制：
1. 从 CheckpointStore 加载指定 iteration 的 LoopState
2. 创建新的 LoopState，复制 conversation 和 global_state
3. 新分支有自己的 STM（从 fork 点开始），但 LTM 是共享的（因为 LTM 是 Arc<LongTermMemory> 引用同一实例）

如果两个分支同时写 LTM → LTM 的 ThreadSafe 包装保证了并发安全（RwLock）。

---

## 第六层：Rust 工程能力

### Q46. sealed enum 层级的设计目的？

**回答框架：**
sealed enum = **sealed trait + private enum variants**，目的是防止外部 crate 新增 enum variant。

好处：
1. **API 稳定性**：外部 crate 不能新增 PermissionLevel 的 variant，所以 match 可以是 exhaustive 的（不需要 `_ => {}` 兜底）
2. **`#[non_exhaustive]` enum** 也防止外部 match exhaustive，但语义不同——non_exhaustive 是"未来可能新增 variant"，sealed 是"只有内部 crate 可以新增"

在哪个模块用了：
- `PermissionLevel`：`#[non_exhaustive]` + sealed（oneai-core）
- `AgentDecision`：`#[non_exhaustive]`（agent_loop.rs:107）
- `NodeAction`：`#[non_exhaustive]`（state_graph.rs:46）
- `EdgeCondition`：`#[non_exhaustive]`

---

### Q47. 为什么还用 async-trait crate 而不是 Rust 原生 async trait？

**回答框架：**
历史原因 + 技术考量：
- OneAI 项目开始时 Rust 原生 async trait 尚未稳定（需要 1.75+）
- `#[async_trait]` 的核心功能：将 async fn 在 trait 中转换为返回 `Pin<Box<dyn Future>>`——这在原生 async trait 中也是类似的实现
- 技术考量：`#[async_trait]` 的 `Send` bounds 更显式——原生 async trait 的 Send bound 语法更复杂（`async fn foo() -> impl Future<Output = T> + Send`）

现在可以迁移到原生 async trait，但工作量较大（19 crate 的所有 trait 定义都需要改），而且功能上没有差异——所以暂时保持 `#[async_trait]`。

---

### Q48. tokio::spawn 在子代理里怎么用？ScopeState 隔离怎么保证？

**回答框架：**
当前状态：SubAgentWrapper **已经使用 tokio::spawn**——竞品分析标注的"未用 tokio::spawn 独立线程"差距已补齐。

具体实现：SubAgentWrapper 在 `spawn_sub_agent()` 中 `tokio::spawn(async { agent_loop.run_with_observer(...) })`，让子 Agent 在独立 async task 中运行。

ScopeState 隔离保证：
- 每个 SubAgent 有自己的 LoopState（独立的 conversation、global_state、iterations）
- SubAgent 的 AgentLoop 是 clone 的（Arc 指针复制），共享 provider/tools/parser/approval_gate 等不变依赖
- 可变状态（conversation）通过 LoopState 的 ownership 隔离——每个 SubAgent 独占自己的 LoopState

**代码证据：** `agent_loop.rs:546-568` 的 Clone impl——所有字段都是 Arc/RwLock/Arc<RwLock>，clone 只复制指针。`sub_agent.rs` 的 SubAgentWrapper 用 tokio::spawn。

---

### Q49. 19 crate workspace 编译时间？feature flag 优化？

**回答框架：**
编译时间：冷编译约 3-5 分钟（依赖项编译为主），增量编译约 10-30 秒（取决于改动范围）。

Feature flag：**当前没有做细粒度 feature flag 优化**——所有 crate 的 feature 基本都是全量开启。这是一个已知的优化方向：为 WASM/RAG/MCP/Postgres 等可选依赖添加 feature gate，让不需要这些功能的编译更快。

编译优化的主要瓶颈：
- `rusqlite` 的 bundled feature 编译 SQLite C 代码（较慢）
- `wasmtime` 的 cranelift JIT 编译器编译较慢

---

### Q50. UniFFI 的 Result<T, E> 映射到 Kotlin/Swift 异常体系？

**回答框架：**
UniFFI 的映射规则：
- Rust `Result<T, OneAIError>` → Kotlin `fun foo(): T; throws OneAIException`
- Rust enum `OneAIError { Config, Provider, Tool }` → Kotlin sealed class `OneAIException.ConfigException`, `ProviderException`, `ToolException`

自定义错误类型的坑：
- Rust enum 的 variant 如果有 `String` 字段 → Kotlin 每个异常类需要对应的 constructor
- `Arc<dyn Trait>` 错误不能导出——需要在 UniFFI 层用具体 struct 包装
- 嵌套 enum（如 `PermissionAction::Deny { reason: String }`）→ Kotlin 需要手写 data class 映射

OneAI 的解决方式：oneai-uniffi crate 的 udl 文件定义所有可导出类型，将复杂的 Rust 类型拆解为 UniFFI 能处理的简单类型。

---

## 第七层：跨界追问

### Q51. 冗余度+多路传输 vs 3层解析防御的思维共性？

**回答框架：**
共性：**不信任单一通道的可靠性，用多层冗余保障结果正确**。

畅连音视频：
- 网络丢包（通道不可靠）→ 冗余发送（同一数据发多路）+ 动态调整冗余度（根据丢包率）
- 是在物理层做冗余

OneAI 3层解析：
- LLM 输出不可靠（模型可能输出格式错误的 JSON）→ 约束解码+模糊修复+回退自纠
- 是在信息层做冗余

本质都是：**假设底层不可靠 → 用多层防御/冗余提高最终可靠性**。这种思维从网络协议设计到安全架构到 LLM 工程是通用的。

---

### Q52. STRIDE 威胁分析 → PermissionLevel + 审批门的方法论延续？

**回答框架：**
方法论延续：
- STRIDE 的 Spoofing（伪造身份）→ Agent 的 **Shell 命令伪装**：恶意 prompt 诱导 Agent 执行危险命令 → deny_by_default 拦截 `rm -rf`
- STRIDE 的 Tampering（篡改数据）→ Agent 的 **文件修改安全**：PermissionLevel 的 Standard 级要求确认才能 edit
- STRIDE 的 Elevation of Privilege（权限提升）→ Agent 的 **权限分级覆写**：deny_by_default 优先级最高防止被低优先级规则绕过

延续的具体体现：PermissionProfile 的 5 级优先级链本质上是 STRIDE 分析的"威胁 → 防御措施"映射——每种权限规则对应一种威胁类型。

---

### Q53. AMS/WMS/PMS 调优 vs ContextBudgetManager 的思维共性？

**回答框架：**
共性：**在有限资源下做优先级调度**。

AMS（Activity Manager Service）：内存不足时优先保留前台 App，后台 App 被杀 → 确保用户体验优先

ContextBudgetManager：上下文预算不足时优先保留最近对话，更早的历史被压缩 → 确保 Agent 的推理质量优先

本质都是：**资源受限 → 按重要性排序 → 保留最重要的，牺牲次要的**。

差异：AMS 的调度是系统级（OS 控制），ContextBudgetManager 是应用级（Agent 自控）。但核心算法思路一样——优先级队列 + 资源预算。

---

### Q54. Multi-Agent 协作场景设计？

**回答框架：**
3 个 Agent 协同完成复杂任务的设计思路：

通信协议：**用 A2A**（不是自定义），因为 A2A 已经定义了 Task-centric 模型（Agent 发送 Task → 对方处理 → 返回 Artifact）。

记忆架构：**共享 LTM + 独立 STM**——每个 Agent 有自己的短期对话上下文，但通过共享的 LTM（vector store）获取全局知识。

冲突解决：
1. **文件冲突**：git worktree 隔离 + merge 策略（和 OneAI 现有 SubAgent 机制一致）
2. **决策冲突**：如果两个 Agent 给出不同建议 → 主 Agent 做 final decision（不是投票，而是综合+判断）
3. **资源冲突**：TokenBudget 总预算分配给三个 Agent（每个子预算独立）

---

### Q55. DomainPack 动态加载新领域——架构怎么改？

**回答框架：**
当前 DomainPack 是静态配置（AppBuilder.build() 时确定），动态加载需要改的地方：

1. **PackRegistry 增加热加载**：从 Git/Local source install + load 在运行时执行（当前已有 install 和 load_installed 方法，但未接入 AgentLoop）
2. **MergedDomainPack 增加动态 merge**：运行时新增 DomainPack → 重新 merge → 更新 AgentLoop 的 domain_pack: Arc<MergedDomainPack>
3. **AgentLoop 的工具集动态更新**：当前 tools 是 `Arc<RwLock<HashMap<String, Arc<dyn Tool>>>>`——RwLock 已经支持运行时修改
4. **范式策略和权限动态生效**：重新 merge 后，新 DomainPack 的范式策略和权限配置需要立即生效

关键挑战：**上下文一致性**——如果新 DomainPack 引入了新上下文源（如新的文件树结构），已经注入的旧上下文和新上下文可能冲突。解决方式：新 DomainPack 激活时做一次 full baseline epoch 重建。

---

### Q56. 加入 Agent 团队第一件事？

**回答框架：**
取决于团队状态：
- **如果团队已有成熟的 Agent 框架**：先用团队框架做实际项目，理解框架的 design choice 和 trade-off，然后把 OneAI 的创新点（DomainPack、PermissionProfile、ContextEpoch）作为 feature contribution 提议——不是替代而是补充
- **如果团队框架不成熟**：评估 OneAI 能否作为团队的基础框架，但需要做实际场景适配（OneAI 目前是实验性项目，缺少生产级打磨）

选择理由：Agent 工程的核心价值不在框架本身，而在**对 Agent 行为的理解和调优**。框架只是工具，重要的是用框架解决实际问题的经验。所以不管用哪个框架，第一步都是做真实项目、积累实战数据。

---

## 红线问题回答要点

### R1. DomainPack 5层各自的 struct 定义长什么样？

**核心字段速答：**
| 层 | struct | 核心字段 |
|---|--------|---------|
| 1 | ToolDecorator | tool_name: String, description_override: String, usage_hint: Option<String> |
| 2 | ContextSource (trait) | key(), load(), refresh_policy(), priority() |
| 3 | PermissionProfile | name, auto_approve: HashSet, require_confirmation: HashSet, deny_by_default: Vec<DenyPattern>, permission_overrides: HashMap, default_threshold: PermissionLevel |
| 4 | ParadigmStrategy | trigger_pattern: String, paradigm_sequence: Vec<DomainParadigmKind>, sub_agent_type: Option<String> |
| 5 | CompressionTemplate | name: String, preserve_patterns: Vec<String>, summary_template: String |

### R2. AgentLoop 一轮迭代的完整代码流

**6 个步骤：**
1. 刷新上下文源 + 拍快照（refresh_sources + take_snapshot + update_snapshot）
2. 组装对话（context_assembler.assemble()）→ 检查是否需要压缩（needs_compression）
3. 注入技能（skill_selector.select_skills）
4. 构建推理请求（build_tool_definitions_for_paradigm + InferenceRequest）
5. 执行推理（provider.infer 或 run_streaming_iteration_async）
6. 解析决策 + 执行工具/委派/切换范式 + 回填结果

### R3. StreamParser 检测 tool_use 的状态机有几种状态？

**4 种状态：**
1. **空闲（idle）**：text_buffer 累积，无 active tool call
2. **ToolIntent（name_detected）**：收到 id+name → 创建 ToolCallBuilder → emit ToolIntentDetected
3. **ArgsAccumulating（args_streaming）**：收到 args 片段 → 追加到 args_buffer → 静默累积
4. **Finalizing（is_final）**：收到 is_final chunk → finalize 所有 pending tool call → emit ToolCallComplete + StreamComplete

转换条件：
- idle → name_detected：收到 ToolCall chunk 且 id 非空
- name_detected → args_streaming：收到 ToolCall chunk 且 id 为空 args 非空
- args_streaming → name_detected（新 tool）：收到 ToolCall chunk 且 id 非空 → 先 finalize 当前再开新
- any → finalizing：收到 is_final chunk

### R4. deny_by_default 为什么优先级最高？

**一句话回答：** fail-safe default（安全兜底原则）——如果 deny 不是最高优先级，auto_approve 可能先批准危险操作，deny 来不及拦截。deny 必须在任何批准逻辑之前检查，确保不可逆操作永远不执行。

### R5. 竞品分析中最关键差距的当前状态？

**所有 Critical 级差距已解决：**
1. ✅ **真实沙箱**：实现了 SandboxBackend trait — Seatbelt（macOS sandbox-exec）+ Docker（Linux 容器）+ Regex（fallback），和 Claude Code/Codex 使用同级别隔离机制
2. ✅ **MCP SSE/StreamableHttp 传输**：三种传输协议（stdio/SSE/StreamableHttp）全部实现，可以直接对接生产级 MCP server

**项目演进状态：** 已推进到 P3-4（Playground/Studio Web UI），P0-P2 全部完成：
- P0: AgentLoop↔Provider闭环 + E2E测试 + MCP完整实现 + SubAgentFactory真实实现
- P1: 原生搜索工具 + Lifecycle Hooks + Interrupt/Resume + StructuredOutput+ModelRetry + WebFetch/NotebookEdit
- P2: Worktree隔离+并行子代理 + StateGraph↔AgentLoop闭环 + OTEL可观测性 + STM↔LTM闭环 + A2A SDK + WASM沙箱引擎

**当前仍需继续推进的方向（P3阶段）：**
- P3-1: API稳定化 + 版本升级0.2.0（✅ 已完成）
- P3-2: DomainPack市场 + PackRegistry（✅ 已完成）
- P3-3: CLI打磨 + clap子命令架构（✅ 已完成）
- P3-4: Playground/Studio Web UI（进行中）
- P3-5: Eval框架 + Benchmark
- P3-6: MCP Server插件生态